<a href="https://colab.research.google.com/github/AgnelFernando/HealthAIAgent/blob/master/notebooks/RAG_Corpus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install psycopg2-binary python-dotenv tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 79.0 MB/s eta 0:00:00


In [13]:
import re
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

def clean_text(s: str) -> str:
    s = s.replace("\u00a0", " ")
    s = s.replace("\\u2013", "-")
    s = s.replace("\\n", "\n")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def chunk_text(text: str, max_tokens: int = 450, overlap_tokens: int = 60):
    text = clean_text(text)
    tokens = enc.encode(text)
    chunks = []
    start = 0
    idx = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_str = enc.decode(chunk_tokens)
        chunks.append((idx, chunk_str, len(chunk_tokens)))
        idx += 1
        start = end - overlap_tokens
        if start < 0:
            start = 0
        if start >= len(tokens) or end >= len(tokens):
            break
    return chunks

In [15]:
from datetime import datetime

def normalize_date(value):
    if not value:
        return None
    value = value.strip()
    try:
        dt = datetime.strptime(value, "%m/%d/%Y")
        return dt.date()
    except ValueError:
        pass

    raise ValueError(f"Unrecognized date format: {value}")

In [26]:
import json

docs = json.load(open("articles.json", "r"))

In [27]:
import psycopg2
from psycopg2.extras import execute_values
from google.colab import userdata

DB_URL = userdata.get('DATABASE_URL')

def insert_docs_and_chunks(docs):
    conn = psycopg2.connect(DB_URL)
    conn.autocommit = False
    try:
        with conn.cursor() as cur:
            for d in docs:
                published_date = normalize_date(d.get("published_date"))
                # 1) Insert doc
                cur.execute(
                    """
                    insert into knowledge_docs (title, source, url, published_date)
                    values (%s, %s, %s, %s)
                    returning id
                    """,
                    (d["title"], d["source"], d.get("url"), d.get("published_date"))
                )
                doc_id = cur.fetchone()[0]

                # 2) Chunk
                chunks = chunk_text(d["text"], max_tokens=450, overlap_tokens=60)

                # 3) Insert chunks
                rows = [
                    (doc_id, chunk_index, chunk_str, tok_est)
                    for (chunk_index, chunk_str, tok_est) in chunks
                ]
                execute_values(
                    cur,
                    """
                    insert into knowledge_chunks (doc_id, chunk_index, chunk_text, tokens_est)
                    values %s
                    """,
                    rows
                )

        conn.commit()
        print("Inserted docs + chunks successfully.")
    except Exception as e:
        conn.rollback()
        raise
    finally:
        conn.close()

insert_docs_and_chunks(docs)

Inserted docs + chunks successfully.


In [28]:
import psycopg2

conn = psycopg2.connect(DB_URL)
with conn.cursor() as cur:
    cur.execute("select count(*) from knowledge_docs;")
    print("knowledge_docs:", cur.fetchone()[0])

    cur.execute("select count(*) from knowledge_chunks;")
    print("knowledge_chunks:", cur.fetchone()[0])

    cur.execute("""
      select d.title, count(c.id) as chunks
      from knowledge_docs d
      join knowledge_chunks c on c.doc_id = d.id
      group by d.title
      order by chunks desc
      limit 5;
    """)
    print(cur.fetchall())
conn.close()

knowledge_docs: 37
knowledge_chunks: 963
[("Sleep's contribution to memory formation", 564), ('Nighttime Sleep Awakening Frequency and Its Consistency Predict Future Academic Performance in College Students', 33), ('Exercise Selection and Common Injuries in Fitness Centers: A Systematic Integrative Review and Practical Recommendations', 33), ('Effect of daridorexant on sleep architecture in patients with chronic insomnia disorder: a pooled post hoc analysis of two randomized phase 3 clinical studies', 30), ('Monitoring Sleep and Nightly Recovery with Wrist-Worn Wearables: Links to Training Load and Performance Adaptations', 27)]
